# Detection Inference Demo

Interactive notebook demonstrating the detection inference pipeline for factory safety cameras.

Covers:
1. Setup and model loading
2. Single-image inference
3. Batch inference with grid visualization
4. Confidence threshold analysis
5. Speed benchmarking
6. Video inference demo

## 1. Setup

Add the pipeline root to `sys.path` and import the predictor.

In [ ]:
import sys
from pathlib import Path

# Ensure pipeline root is importable
sys.path.insert(0, str(Path("..").resolve()))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
import time

from utils.config import load_config
from predictor import DetectionPredictor
from video_inference import VideoProcessor

# For inline plots
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 8)

print("Imports OK")

### Configuration

Edit these paths to point to your model and data config.

In [ ]:
# --- USER CONFIG: edit these paths ---

# Path to trained model (.pt or .onnx)
MODEL_PATH = "../../03_development/models/fire_yoloxm_640_v1.onnx"  # example

# Data config (provides class names, input size, normalization)
DATA_CONFIG_PATH = "../features/safety-fire_detection/configs/01_data.yaml"  # example: fire detection

# Test images directory (or list of paths)
TEST_IMAGES_DIR = "../../dataset_store/fire_detection/test/images"  # example

# Test video (optional)
TEST_VIDEO_PATH = None  # set to a video path if available

# Inference settings
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.45

In [ ]:
# Load data config
data_config = load_config(DATA_CONFIG_PATH)
print(f"Dataset: {data_config['dataset_name']}")
print(f"Classes: {data_config['names']}")
print(f"Input size: {data_config['input_size']}")
print(f"Num classes: {data_config['num_classes']}")

In [ ]:
# Create predictor
predictor = DetectionPredictor(
    model_path=MODEL_PATH,
    data_config=data_config,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
)
print(f"Backend: {predictor.backend}")
print(f"Device: {predictor.device}")
print(f"Classes: {predictor.class_names}")

## 2. Single-Image Inference

Load one image, run detection, and visualize the results.

In [ ]:
def show_bgr(image, title="", figsize=(10, 8)):
    """Display a BGR image in matplotlib (converts to RGB)."""
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Find test images
test_dir = Path(TEST_IMAGES_DIR)
if test_dir.exists():
    image_paths = sorted(test_dir.glob("*.jpg"))[:20]  # first 20
    if not image_paths:
        image_paths = sorted(test_dir.glob("*.png"))[:20]
    print(f"Found {len(image_paths)} test images")
else:
    print(f"Test directory not found: {test_dir}")
    print("Set TEST_IMAGES_DIR to a valid path, or manually specify image_paths.")
    image_paths = []

In [ ]:
# Run inference on the first image
if image_paths:
    img_path = image_paths[0]
    print(f"Image: {img_path.name}")
    
    results = predictor.predict_file(str(img_path))
    print(f"Detections: {len(results['boxes'])}")
    for i in range(len(results['boxes'])):
        box = results['boxes'][i]
        print(f"  [{results['class_names'][i]}] conf={results['scores'][i]:.3f} "
              f"box=[{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]")
    
    # Visualize
    image = cv2.imread(str(img_path))
    annotated = predictor.visualize(image, results)
    show_bgr(annotated, f"{img_path.name} — {len(results['boxes'])} detections")
else:
    print("No test images available. Skipping.")

## 3. Batch Inference

Process multiple images and display results in a grid.

In [ ]:
def show_grid(images, titles, cols=4, figsize=(20, 12)):
    """Display a grid of BGR images."""
    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1:
        axes = [axes] if cols == 1 else list(axes)
    else:
        axes = [ax for row in axes for ax in row]
    
    for i, (img, title) in enumerate(zip(images, titles)):
        axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(title, fontsize=9)
        axes[i].axis("off")
    
    # Hide unused subplots
    for j in range(n, len(axes)):
        axes[j].axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
if len(image_paths) >= 2:
    batch_paths = image_paths[:8]  # up to 8 images
    batch_images = [cv2.imread(str(p)) for p in batch_paths]
    batch_images = [img for img in batch_images if img is not None]
    
    print(f"Running batch inference on {len(batch_images)} images...")
    t0 = time.time()
    batch_results = predictor.predict_batch(batch_images)
    t1 = time.time()
    print(f"Batch inference: {t1 - t0:.3f}s ({len(batch_images) / (t1 - t0):.1f} img/s)")
    
    # Visualize grid
    annotated_imgs = []
    titles = []
    for img, res, path in zip(batch_images, batch_results, batch_paths):
        vis = predictor.visualize(img, res)
        annotated_imgs.append(vis)
        titles.append(f"{path.name} ({len(res['boxes'])} det)")
    
    show_grid(annotated_imgs, titles, cols=4)
else:
    print("Need at least 2 images for batch demo. Skipping.")

## 4. Confidence Threshold Analysis

Compare detection results at different confidence thresholds on the same image.
This helps find the optimal trade-off between precision and recall.

In [ ]:
if image_paths:
    # Use a detection-rich image (first one with detections, or just the first)
    test_image = cv2.imread(str(image_paths[0]))
    thresholds = [0.1, 0.25, 0.5, 0.7, 0.9]
    
    vis_list = []
    title_list = []
    
    for thresh in thresholds:
        # Temporarily override threshold
        original_conf = predictor.conf_threshold
        predictor.conf_threshold = thresh
        
        res = predictor.predict(test_image)
        vis = predictor.visualize(test_image, res)
        vis_list.append(vis)
        title_list.append(f"conf={thresh} ({len(res['boxes'])} det)")
        
        predictor.conf_threshold = original_conf
    
    show_grid(vis_list, title_list, cols=3, figsize=(18, 10))
    
    # Plot detection count vs threshold
    counts = [int(t.split("(")[1].split(" ")[0]) for t in title_list]
    plt.figure(figsize=(8, 4))
    plt.bar([str(t) for t in thresholds], counts, color="steelblue")
    plt.xlabel("Confidence Threshold")
    plt.ylabel("Number of Detections")
    plt.title("Detections vs Confidence Threshold")
    plt.tight_layout()
    plt.show()
else:
    print("No test images available. Skipping.")

## 5. Speed Benchmark

Measure per-image inference time over multiple runs to get a reliable estimate.
Results include preprocessing + forward pass + NMS postprocessing.

In [ ]:
if image_paths:
    test_image = cv2.imread(str(image_paths[0]))
    
    # Warmup (important for GPU and ONNX Runtime)
    print("Warming up (5 iterations)...")
    for _ in range(5):
        _ = predictor.predict(test_image)
    
    # Benchmark
    num_runs = 50
    times = []
    print(f"Benchmarking ({num_runs} iterations)...")
    for _ in range(num_runs):
        t0 = time.perf_counter()
        _ = predictor.predict(test_image)
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)  # ms
    
    times = np.array(times)
    print(f"\nResults ({num_runs} runs):")
    print(f"  Mean:   {times.mean():.1f} ms")
    print(f"  Median: {np.median(times):.1f} ms")
    print(f"  Std:    {times.std():.1f} ms")
    print(f"  Min:    {times.min():.1f} ms")
    print(f"  Max:    {times.max():.1f} ms")
    print(f"  FPS:    {1000 / times.mean():.1f}")
    
    # Plot distribution
    plt.figure(figsize=(8, 4))
    plt.hist(times, bins=20, color="steelblue", edgecolor="white")
    plt.axvline(times.mean(), color="red", linestyle="--", label=f"Mean: {times.mean():.1f}ms")
    plt.axvline(np.median(times), color="orange", linestyle="--", label=f"Median: {np.median(times):.1f}ms")
    plt.xlabel("Inference Time (ms)")
    plt.ylabel("Count")
    plt.title(f"Inference Latency Distribution ({predictor.backend})")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No test images available. Skipping.")

## 6. Video Demo

Process a short video clip with detection and alert logic.
Set `TEST_VIDEO_PATH` in the config cell above.

In [ ]:
if TEST_VIDEO_PATH is not None and Path(TEST_VIDEO_PATH).exists():
    print(f"Processing video: {TEST_VIDEO_PATH}")
    
    # Create video processor with default alert config
    processor = VideoProcessor(predictor)
    
    # Output path for annotated video
    video_path = Path(TEST_VIDEO_PATH)
    output_path = video_path.parent / f"{video_path.stem}_annotated.mp4"
    
    # Process (skip every other frame for speed)
    summary = processor.process_video(
        video_path=TEST_VIDEO_PATH,
        output_path=str(output_path),
        skip_frames=1,  # process every 2nd frame
    )
    
    print(f"\n--- Video Summary ---")
    print(f"Total frames:    {summary['total_frames']}")
    print(f"Processed:       {summary['processed_frames']}")
    print(f"Detections:      {summary['total_detections']}")
    print(f"Alerts:          {summary['alert_count']}")
    print(f"Processing FPS:  {summary['fps']}")
    print(f"Elapsed:         {summary['elapsed_seconds']}s")
    print(f"Class counts:    {summary['class_counts']}")
    
    if summary['alerts']:
        print(f"\n--- Alerts ---")
        for alert in summary['alerts']:
            print(f"  Frame {alert['frame_idx']}: {alert['message']}")
    
    print(f"\nAnnotated video saved to: {output_path}")
else:
    print("No video path set or file not found. Skipping video demo.")
    print("Set TEST_VIDEO_PATH in the config cell above to enable this section.")

---

## Summary

This notebook demonstrated:
- Loading a detection model (PyTorch or ONNX) with `DetectionPredictor`
- Single-image and batch inference
- Confidence threshold sweep to find the best operating point
- Latency benchmarking for deployment planning
- Video processing with frame-level alert logic via `VideoProcessor`

Next steps:
- Tune `conf_threshold` per model based on the threshold analysis
- Use `video_inference.py` alert config to match project requirements
- Export to ONNX for edge deployment benchmarking